# Session 1: Linear Regression from Scratch with NumPy

## Overview

A power plant has to tell the electricity grid, hours in advance, how much power it will deliver. For a
combined-cycle gas plant the biggest factor is the weather: hot air is thin, so the turbines push less of it
and produce less power. In this session you build the simplest possible forecast, **a straight line from air
temperature to power output**, and you build it by hand: the model, the cost function and the **gradient
descent** loop that fits it. Then you check your answer against the exact formula and scikit-learn.

By the end, you will be able to:

- Write a simple linear regression model and its mean squared error (MSE) cost in NumPy.
- Derive and code the gradients, and fit the model with gradient descent.
- Choose a learning rate, recognise divergence, and explain why scaling the input speeds up training.
- Check a hand-built model against the closed-form solution and `sklearn.linear_model.LinearRegression`.
- Evaluate a regression model on held-out data with MAE, RMSE and R², against a baseline.

**Prerequisites:** basic Python and NumPy ([2. Python Intro and Math Foundation](../2.%20Python%20Intro%20and%20Math%20Foundation));
derivatives at the level of "the slope of a curve". [6. Statistical Foundations](../6.%20Statistical%20Foundations%20for%20Data%20Science),
Session 9, introduces the same line from the statistics side.

**Dataset:** [UCI Combined Cycle Power Plant](https://archive.ics.uci.edu/dataset/294/combined+cycle+power+plant).
9,568 hourly averages (2006-2011) from a gas-and-steam power plant running at full load: ambient temperature,
exhaust vacuum, ambient pressure, relative humidity, and the net electrical output in megawatts (MW). It is
downloaded automatically by the code.

## Where this fits

This module teaches the core machine learning algorithms by building something real with each one.

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models **← you are here** |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** How well can a straight line forecast the plant's output from the temperature alone?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| numpy | The model, cost and gradients, by hand |
| pandas | Tables |
| scikit-learn | Train/test split, the reference model, metrics |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from importlib.metadata import version

for package in ["ucimlrepo", "numpy", "pandas", "scikit-learn", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the data and set aside a test set

The cell downloads the data, gives the columns readable names, and puts 20% of the hours aside as a **test
set**. The model never sees these hours while it is being fitted; at the end they tell us how well it forecasts
hours it has not seen. `random_state=42` makes the split the same on every run.

In [ ]:
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

plant = fetch_ucirepo(id=294).data.original.rename(columns={
    "AT": "temperature", "V": "vacuum", "AP": "pressure", "RH": "humidity", "PE": "output"})
train, test = train_test_split(plant, test_size=0.2, random_state=42)

print(f"{len(plant):,} hours: {len(train):,} to fit the model, {len(test):,} held out")
print(train.describe().round(1).T[["mean", "std", "min", "max"]])

**Read the output:** 7,654 hours to fit, 1,914 held out. Temperatures range from about 2 °C to 37 °C, and the
output from about 420 MW to 496 MW. The average hour delivers about 454 MW, with a standard deviation of 17 MW.

## Step 3: Look at the relationship

Before fitting any model, plot the input against the target. A straight line only makes sense if the points
roughly follow one.

In [ ]:
x_train, y_train = train["temperature"].to_numpy(), train["output"].to_numpy()
x_test, y_test = test["temperature"].to_numpy(), test["output"].to_numpy()

print(f"correlation between temperature and output: {np.corrcoef(x_train, y_train)[0, 1]:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_train, y_train, s=3, alpha=0.3)
ax.set_xlabel("ambient temperature (°C)")
ax.set_ylabel("net output (MW)")
ax.set_title("Hotter air, less power")
plt.show()

**Read the output:** the correlation is -0.948, very strong and negative, and the cloud of points is close to a
straight band falling from left to right. A straight line is a sensible first model. The band has some width,
though: at any one temperature, output still varies by about ±10 MW, so even a perfect line cannot be exact.

## Step 4: The model and its cost

A simple linear regression predicts

$$\hat y = m x + b$$

with a **slope** $m$ (MW per °C) and an **intercept** $b$ (MW at 0 °C). To say how good a choice of $m$ and $b$
is, we use the **mean squared error** over the $n$ training hours:

$$J(m, b) = \frac{1}{n}\sum_{i=1}^{n}\big(y_i - (m x_i + b)\big)^2$$

Squaring makes every error count as positive and punishes big misses more than small ones. Fitting the model
means finding the $m$ and $b$ with the lowest cost.

In [ ]:
def predict(x, m, b):
    return m * x + b


def cost(x, y, m, b):
    return np.mean((y - predict(x, m, b)) ** 2)


for m, b in [(0, 0), (0, y_train.mean()), (-1, 480), (-2, 490)]:
    print(f"m = {m:5.1f}, b = {b:6.1f}  ->  cost = {cost(x_train, y_train, m, b):12,.1f}")

**Read the output:** the line through zero is hopeless (cost about 207,000). A flat line at the average output,
which ignores the temperature completely, already brings the cost down to 291.5; this is simply the variance of
the output. Guessing a downward slope does much better again: m = -2, b = 490 costs 44.8. Gradient descent finds
the best values for us instead of guessing.

## Step 5: The gradients

The **gradient** says how the cost changes when we nudge each parameter. Differentiating $J$:

$$\frac{\partial J}{\partial m} = -\frac{2}{n}\sum_{i} x_i\,(y_i - \hat y_i) \qquad
\frac{\partial J}{\partial b} = -\frac{2}{n}\sum_{i} (y_i - \hat y_i)$$

**Gradient descent** repeatedly takes a small step *against* the gradient, downhill:

$$m \leftarrow m - \alpha\,\frac{\partial J}{\partial m} \qquad b \leftarrow b - \alpha\,\frac{\partial J}{\partial b}$$

The step size $\alpha$ is the **learning rate**, a setting we choose (a *hyperparameter*), not something the
model learns.

In [ ]:
def gradients(x, y, m, b):
    error = y - predict(x, m, b)
    return -2 * np.mean(x * error), -2 * np.mean(error)


def gradient_descent(x, y, learning_rate, n_steps, m=0.0, b=0.0):
    history = []
    for _ in range(n_steps):
        dm, db = gradients(x, y, m, b)
        m, b = m - learning_rate * dm, b - learning_rate * db
        history.append(cost(x, y, m, b))
    return m, b, history


dm, db = gradients(x_train, y_train, 0.0, 0.0)
print(f"gradient at m=0, b=0:  dJ/dm = {dm:,.0f}   dJ/db = {db:,.0f}")

**Read the output:** both gradients are large and negative: increasing either m or b from zero would lower the
cost, so the descent step increases both. Notice that dJ/dm is about 20 times larger than dJ/db. The gradient for
m is multiplied by x, and temperatures are around 20. Keep this in mind for Step 6.

## Step 6: Choosing a learning rate

Run gradient descent on the raw temperatures with three learning rates.

In [ ]:
for learning_rate in [0.005, 0.001, 0.0001]:
    with np.errstate(over="ignore", invalid="ignore"):     # silence overflow warnings when it diverges
        m, b, history = gradient_descent(x_train, y_train, learning_rate, n_steps=1000)
    print(f"learning rate {learning_rate:<7}  cost after 1,000 steps = {history[-1]:12.4g}   "
          f"m = {m:9.3g}, b = {b:9.3g}")

**Read the output:**

- **0.005 diverges:** the result is `nan` ("not a number"). Each step overshoots the bottom of the valley and lands
  higher up the other side, so the steps grow until the numbers overflow.
- **0.001 is stable but hopeless:** after 1,000 steps the cost is still 18,760, far worse than the flat line at the
  average (291.5). The intercept has crawled from 0 to only 111 (its best value is about 497, Step 8). With the
  intercept stuck that low, the best the slope can do is tilt the line *upwards* through the cloud: m = +15, the
  wrong sign.
- **0.0001** is safe, and even slower.

This is the classic learning-rate dilemma. The cost is a long, narrow valley: very steep in the m direction
(because x is large) and very flat in the b direction. Any step large enough to make progress in b overshoots in
m; any step small enough for m makes almost no progress in b.

## Step 7: Scale the input, then descend

The fix is to **standardise** the input: subtract its mean and divide by its standard deviation, so it is centred
on 0 with a spread of 1. Then m and b have similar steepness and one learning rate suits both. The mean and
standard deviation come from the **training** set only; the test set must be scaled with the same numbers.

In [ ]:
mean, std = x_train.mean(), x_train.std()
z_train, z_test = (x_train - mean) / std, (x_test - mean) / std

m_z, b_z, history = gradient_descent(z_train, y_train, learning_rate=0.1, n_steps=100)
print(f"after 100 steps: cost = {history[-1]:.4f}, m = {m_z:.4f} MW per standard deviation, b = {b_z:.4f}")

# Convert back to the original units: y = m_z * (x - mean) / std + b_z
m_hat, b_hat = m_z / std, b_z - m_z * mean / std
print(f"in °C:           output = {b_hat:.2f} {m_hat:+.4f} * temperature")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history)
ax.set_yscale("log")
ax.set_xlabel("step")
ax.set_ylabel("cost (MSE, log scale)")
ax.set_title("Gradient descent on the scaled input")
plt.show()

**Read the output:** with the scaled input, a learning rate of 0.1 reaches a cost of 29.56 in just 100 steps;
the raw input was still at 18,760 after 1,000. The cost curve falls steeply and then flattens as the steps reach
the bottom of the valley.

Converted back to the original units, the model is **output = 497.15 - 2.18 × temperature**: each extra degree
costs the plant about 2.2 MW. The intercept is the forecast at 0 °C.

Scaling changes nothing about the answer, only how fast we find it. Every gradient-based model in this module
(and neural networks) relies on it.

## Step 8: Check against the exact answer

For a straight line, the best m and b also have a **closed-form** solution (Statistical Foundations, Session 9):

$$m = \frac{\text{Cov}(x, y)}{\text{Var}(x)} \qquad b = \bar y - m\,\bar x$$

scikit-learn's `LinearRegression` computes the same least-squares solution. All three should agree.

In [ ]:
from sklearn.linear_model import LinearRegression

m_exact = np.cov(x_train, y_train, bias=True)[0, 1] / np.var(x_train)
b_exact = y_train.mean() - m_exact * x_train.mean()
sk = LinearRegression().fit(x_train.reshape(-1, 1), y_train)

print(pd.DataFrame({"slope m": [m_hat, m_exact, sk.coef_[0]],
                    "intercept b": [b_hat, b_exact, sk.intercept_]},
                   index=["gradient descent", "closed form", "scikit-learn"]).round(4))

**Read the output:** all three agree to four decimal places: slope -2.1771, intercept 497.1542. Your hand-written
gradient descent found the true least-squares line.

Why learn gradient descent at all, if a formula exists? Because the formula only exists for a few models. Logistic
regression (Session 5), neural networks, and most modern models have no closed form and are fitted by some
version of exactly this loop.

`reshape(-1, 1)` turns the 1-D array into a table with one column: scikit-learn always expects inputs as a 2-D
table (rows × features), even when there is only one feature.

## Step 9: Evaluate on the held-out hours

Now the model forecasts the 1,914 test hours it has never seen. Three standard regression metrics:

- **MAE** (mean absolute error): the average miss, in MW.
- **RMSE** (root mean squared error): like MAE, but big misses count more. Same units as the target.
- **R²**: the share of the variation in output the model explains. 0 means no better than always predicting the
  average; 1 is perfect.

A metric means little on its own, so we compare against a **baseline** that ignores the temperature and always
predicts the training average.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def report(name, y_true, y_pred):
    print(f"{name:22s} MAE = {mean_absolute_error(y_true, y_pred):5.2f} MW   "
          f"RMSE = {mean_squared_error(y_true, y_pred) ** 0.5:5.2f} MW   R² = {r2_score(y_true, y_pred):6.3f}")


report("baseline (average)", y_test, np.full(len(y_test), y_train.mean()))
report("straight line", y_test, predict(x_test, m_hat, b_hat))

**Read the output:** always predicting the average misses by 14.8 MW on average (its R² is 0, by definition).
The straight line cuts that to 4.3 MW (RMSE 5.4 MW) and explains 90% of the variation (R² = 0.900). For one input and two numbers, that is a
strong forecast.

The test R² is close to what the correlation in Step 3 promised (0.948² = 0.90), because the test hours come from
the same plant and the same years as the training hours.

## Step 10: Look at the errors

A good-looking R² can hide a pattern. Plot the **residuals** (actual minus predicted) against the temperature. If
the line captures the relationship, they should scatter evenly around zero with no shape.

In [ ]:
residuals = y_test - predict(x_test, m_hat, b_hat)
bands = pd.cut(x_test, bins=[0, 10, 15, 20, 25, 30, 40])
print(pd.Series(residuals).groupby(bands, observed=True).agg(["mean", "std", "count"]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
order = np.argsort(x_test)
axes[0].scatter(x_test, y_test, s=3, alpha=0.3, label="actual")
axes[0].plot(x_test[order], predict(x_test[order], m_hat, b_hat), color="tab:red", label="fitted line")
axes[0].set_xlabel("temperature (°C)")
axes[0].set_ylabel("output (MW)")
axes[0].legend()
axes[1].scatter(x_test, residuals, s=3, alpha=0.3)
axes[1].axhline(0, color="black")
axes[1].set_xlabel("temperature (°C)")
axes[1].set_ylabel("residual (MW)")
axes[1].set_title("Residuals")
plt.show()

**Read the output:** the residuals are not pure noise. The mean residual changes with the temperature: the line
**over-forecasts** between 20 and 25 °C (mean residual -2.4 MW) and clearly **under-forecasts** on the hottest
days, above 30 °C (+4.8 MW). The true relationship bends, and a straight line cannot bend. The hottest days are
exactly when the grid needs power most, so this error matters.

Within each band the residuals still spread by about ±5 MW (standard deviation 4.5-5.4 MW). The temperature does
not explain everything: the other three weather inputs still carry information. Session 2 adds them.

## What this session hands to the next

- **A working gradient descent**, written by hand: model, cost, gradients, update loop.
- **Two practical rules:** pick the learning rate by watching the cost, and scale the inputs first.
- **A benchmark:** one input gives test RMSE 5.4 MW and R² 0.90. Session 2 uses all four inputs and must beat it.
- **A known weakness:** the residuals show a slight curve the straight line misses.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Learning rate too large | Cost grows every step, then `inf` or `nan` | Divide the learning rate by 10 until the cost falls steadily |
| Learning rate too small | Cost falls, but the parameters are far from their best values | Scale the input; run more steps; plot the cost curve |
| Not scaling the input | One parameter converges, the other crawls | Standardise: `(x - mean) / std` |
| Scaling with test-set statistics | Information from the test set leaks into the model | Compute mean and std on the training set; apply them to both |
| Passing a 1-D array to scikit-learn | `ValueError: Expected 2D array` | `x.reshape(-1, 1)` or select a one-column DataFrame: `df[["temperature"]]` |
| Judging the model by training error | Looks better than it really is | Report metrics on the held-out test set, next to a baseline |
| Trusting R² alone | Hides systematic errors | Plot the residuals |

## Exercises

- [ ] In Step 6, try learning rates 0.002, 0.003 and 0.004. Where exactly does divergence start?
- [ ] In Step 7, try learning rates of 0.5, 1.0 and 1.1. Which converge, and how fast? Explain what happens at 1.0.
- [ ] Replace the temperature with `vacuum` (exhaust vacuum) as the only input. Is it a better or a worse
  predictor? Check the correlation first, then the test RMSE.
- [ ] Change the cost to the mean *absolute* error and derive its gradient (hint: the derivative of |e| is the sign
  of e). Does gradient descent still find the same line?

Next, Session 2 uses all four weather inputs with scikit-learn: multiple linear regression, interpreting
coefficients, and choosing which inputs to keep.